# Verify Miller et al. (2026) MEK isoform model variants

This notebook verifies the curated BNGL files at three levels:

1. Re-run BioNetGen and compare generated outputs with committed reference files.
2. Independently integrate the generated mass-action network with SciPy and compare observables with BioNetGen.
3. Compare curated simulations with reported data from the source material.

The reported-data checks compare the WT simulated scaled outputs with the Miller et al. PyBNF quantitative WT table and compare the Kocieniewski and Lipniacki (2013) WT simulation with digitized Figure 4 model-prediction curves.


In [1]:
from __future__ import annotations

import os
import shutil
import subprocess
import tempfile
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy.spatial import cKDTree

MODEL_DIR = Path.cwd()
if not (MODEL_DIR / "metadata.yaml").exists():
    MODEL_DIR = Path("models/mek_isoform_erk_feedback_miller2026").resolve()
REPO_ROOT = MODEL_DIR.parents[1]
REF_DIR = MODEL_DIR / "reference"

MODEL_STEMS = [
    "mek_isoform_erk_feedback_miller2026",
    "mek_isoform_erk_feedback_miller2026_kocieniewski2013",
]
LABELS = {
    "mek_isoform_erk_feedback_miller2026": "Miller 2026 MLE",
    "mek_isoform_erk_feedback_miller2026_kocieniewski2013": "Kocieniewski 2013",
}

def bionetgen_command(model_name: str):
    env_bng2 = os.environ.get("BNG2_PL")
    if env_bng2:
        return [env_bng2, model_name], None

    local_bng2 = Path.home() / "Simulations" / "BioNetGen-2.9.3" / "BNG2.pl"
    if local_bng2.exists():
        return [str(local_bng2), model_name], str(local_bng2.parent)

    repo_bng2 = REPO_ROOT / ".venv/lib/python3.12/site-packages/bionetgen/bng-mac/BNG2.pl"
    if repo_bng2.exists():
        return [str(repo_bng2), model_name], str(repo_bng2.parent)

    cli = shutil.which("bionetgen")
    if cli:
        return [cli, "run", model_name], None

    uv = shutil.which("uv")
    if uv:
        return [
            uv,
            "run",
            "--with",
            "bionetgen",
            "--with",
            "setuptools<81",
            "bionetgen",
            "run",
            model_name,
        ], None

    raise RuntimeError("BioNetGen not found. Set BNG2_PL, install bionetgen, or install uv.")


print(f"Model directory: {MODEL_DIR}")
print(f"BioNetGen command prefix: {bionetgen_command('MODEL.bngl')[0][0]}")

Model directory: /Users/wish/Code/BNGL-Models/models/mek_isoform_erk_feedback_miller2026
BioNetGen command prefix: /Users/wish/Simulations/BioNetGen-2.9.3/BNG2.pl


Matplotlib is building the font cache; this may take a moment.


In [2]:
def table_from_gdat(path: Path) -> pd.DataFrame:
    header = path.read_text().splitlines()[0].lstrip("#").split()
    data = np.loadtxt(path, comments="#")
    return pd.DataFrame(data, columns=header)


def numeric_max_errors(a: pd.DataFrame, b: pd.DataFrame) -> dict[str, float]:
    common = [c for c in a.columns if c in b.columns]
    av = a[common].to_numpy(dtype=float)
    bv = b[common].to_numpy(dtype=float)
    abs_err = np.max(np.abs(av - bv))
    rel_err = np.max(np.abs(av - bv) / np.maximum(1.0, np.abs(bv)))
    return {"max_abs": float(abs_err), "max_rel_or_abs": float(rel_err)}


def normalized_net_text(path: Path) -> str:
    lines = []
    for line in path.read_text().splitlines():
        if line.startswith("#"):
            continue
        lines.append(" ".join(line.split()))
    return "\n".join(lines)


def run_bionetgen(model_file: Path, workdir: Path) -> None:
    shutil.copy2(model_file, workdir / model_file.name)
    cmd, bngpath = bionetgen_command(model_file.name)
    env = os.environ.copy()
    if bngpath is not None:
        env["BNGPATH"] = bngpath
    subprocess.run(
        cmd,
        cwd=workdir,
        env=env,
        check=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )


reference_rows = []
with tempfile.TemporaryDirectory(prefix="miller2026_bng_") as tmp:
    tmp_path = Path(tmp)
    for stem in MODEL_STEMS:
        run_bionetgen(MODEL_DIR / f"{stem}.bngl", tmp_path)
        net_match = normalized_net_text(tmp_path / f"{stem}_ode.net") == normalized_net_text(
            REF_DIR / f"{stem}_ode.net"
        )
        gdat_err = numeric_max_errors(
            table_from_gdat(tmp_path / f"{stem}_ode.gdat"),
            table_from_gdat(REF_DIR / f"{stem}_ode.gdat"),
        )
        cdat_err = numeric_max_errors(
            table_from_gdat(tmp_path / f"{stem}_ode.cdat"),
            table_from_gdat(REF_DIR / f"{stem}_ode.cdat"),
        )
        reference_rows.append(
            {
                "model": LABELS[stem],
                "ode_net_exact": net_match,
                "gdat_max_rel_or_abs": gdat_err["max_rel_or_abs"],
                "cdat_max_rel_or_abs": cdat_err["max_rel_or_abs"],
            }
        )

reference_check = pd.DataFrame(reference_rows)
reference_check

Out[0]: 
               model  ode_net_exact  gdat_max_rel_or_abs  cdat_max_rel_or_abs
0    Miller 2026 MLE           True                  0.0                  0.0
1  Kocieniewski 2013           True                  0.0                  0.0


In [3]:
def section(lines: list[str], name: str) -> list[str]:
    start = lines.index(f"begin {name}\n") + 1
    end = lines.index(f"end {name}\n")
    return lines[start:end]


def safe_eval(expr: str, env: dict[str, float]) -> float:
    return float(eval(expr.replace("^", "**"), {"__builtins__": {}}, env))


def parse_terms(expr: str) -> list[tuple[float, int]]:
    terms = []
    for term in expr.split(","):
        if "*" in term:
            coef, idx = term.split("*")
            terms.append((float(coef), int(idx) - 1))
        else:
            terms.append((1.0, int(term) - 1))
    return terms


def parse_net(path: Path):
    lines = path.read_text().splitlines(True)
    params: dict[str, float] = {}
    for line in section(lines, "parameters"):
        raw = line.split("#", 1)[0].strip()
        if raw:
            _, name, expr = raw.split(None, 2)
            params[name] = safe_eval(expr, params)

    for line in section(lines, "functions"):
        raw = line.split("#", 1)[0].strip()
        if raw:
            _, name, expr = raw.split(None, 2)
            name = name.removesuffix("()")
            try:
                params[name] = safe_eval(expr, params)
            except NameError:
                pass

    y0 = []
    for line in section(lines, "species"):
        raw = line.split("#", 1)[0].strip()
        if raw:
            parts = raw.split()
            idx = int(parts[0])
            y0.append(safe_eval(parts[-1], params))
            assert idx == len(y0)

    reactions = []
    for line in section(lines, "reactions"):
        raw = line.split("#", 1)[0].strip()
        if raw:
            parts = raw.split()
            reactants = [] if parts[1] == "0" else [int(x) - 1 for x in parts[1].split(",")]
            products = [] if parts[2] == "0" else [int(x) - 1 for x in parts[2].split(",")]
            rate = safe_eval("".join(parts[3:]), params)
            reactions.append((reactants, products, rate))

    groups = {}
    for line in section(lines, "groups"):
        raw = line.split("#", 1)[0].strip()
        if raw:
            _, name, expr = raw.split(None, 2)
            groups[name] = parse_terms(expr)

    return np.array(y0, dtype=float), reactions, groups, params


def integrate_network(net_path: Path, t_eval: np.ndarray):
    y0, reactions, groups, params = parse_net(net_path)
    n_species = len(y0)
    n_reactions = len(reactions)
    stoich = np.zeros((n_species, n_reactions))
    reactant_lists = []
    rates = np.zeros(n_reactions)

    for j, (reactants, products, rate) in enumerate(reactions):
        rates[j] = rate
        reactant_lists.append(reactants)
        for i in products:
            stoich[i, j] += 1.0
        for i in reactants:
            stoich[i, j] -= 1.0

    def rhs(_t, y):
        y_nonnegative = np.maximum(y, 0.0)
        flux = rates.copy()
        for j, reactants in enumerate(reactant_lists):
            for i in reactants:
                flux[j] *= y_nonnegative[i]
        return stoich @ flux

    sol = solve_ivp(
        rhs,
        (float(t_eval[0]), float(t_eval[-1])),
        y0,
        t_eval=t_eval,
        method="LSODA",
        rtol=1e-8,
        atol=1e-8,
    )
    assert sol.success, sol.message
    return sol.y, groups, params


def group_series(y: np.ndarray, terms: list[tuple[float, int]]) -> np.ndarray:
    return sum(coef * y[idx] for coef, idx in terms)


independent_rows = []
independent_curves = {}
for stem in MODEL_STEMS:
    bng = table_from_gdat(REF_DIR / f"{stem}_ode.gdat")
    y, groups, params = integrate_network(REF_DIR / f"{stem}_ode.net", bng["time"].to_numpy())
    curves = {
        "pSos1_wt": group_series(y, groups["pSos1_wt"]),
        "pEGFR_wt": group_series(y, groups["pEGFR_wt"]),
        "pERK1_2_wt": group_series(y, groups["pERK1_2_wt"]),
        "MEK_pRDS": group_series(y, groups["MEK_pRDS"]),
    }
    curves["scaled_pSOS1"] = curves["pSos1_wt"] * params["scalepSos1"]
    curves["scaled_pEGFR"] = curves["pEGFR_wt"] * params["scalepEGFR"]
    curves["scaled_pERK"] = curves["pERK1_2_wt"] * params["scalepERK"]
    independent_curves[stem] = curves

    for obs in ["pSos1_wt", "pEGFR_wt", "pERK1_2_wt", "MEK_pRDS", "scaled_pERK"]:
        ref = bng[obs].to_numpy()
        err = np.max(np.abs(curves[obs] - ref) / np.maximum(1.0, np.abs(ref)))
        independent_rows.append(
            {"model": LABELS[stem], "observable": obs, "max_rel_or_abs": float(err)}
        )

independent_check = pd.DataFrame(independent_rows)
independent_check

Out[0]: 
               model   observable  max_rel_or_abs
0    Miller 2026 MLE     pSos1_wt    2.905868e-07
1    Miller 2026 MLE     pEGFR_wt    1.701445e-08
2    Miller 2026 MLE   pERK1_2_wt    1.324675e-06
3    Miller 2026 MLE     MEK_pRDS    8.620232e-08
4    Miller 2026 MLE  scaled_pERK    1.730237e-07
5  Kocieniewski 2013     pSos1_wt    1.636109e-07
6  Kocieniewski 2013     pEGFR_wt    6.086976e-09
7  Kocieniewski 2013   pERK1_2_wt    6.950803e-07
8  Kocieniewski 2013     MEK_pRDS    9.077907e-08
9  Kocieniewski 2013  scaled_pERK    2.041143e-08


In [4]:
reported = pd.read_csv(REF_DIR / "miller2026_wt_exp.csv")
reported_rows = []
for stem in MODEL_STEMS:
    bng = table_from_gdat(REF_DIR / f"{stem}_ode.gdat")
    for col in ["scaled_pSOS1", "scaled_pEGFR", "scaled_pERK"]:
        pred = np.interp(reported["time"], bng["time"], bng[col])
        obs = reported[col].to_numpy(dtype=float)
        mask = ~np.isnan(obs)
        rmse = np.sqrt(np.mean((pred[mask] - obs[mask]) ** 2))
        data_range = np.nanmax(obs) - np.nanmin(obs)
        reported_rows.append(
            {
                "model": LABELS[stem],
                "observable": col,
                "rmse_AU": float(rmse),
                "rmse_over_range": float(rmse / data_range),
            }
        )
reported_check = pd.DataFrame(reported_rows)
reported_check

Out[0]: 
               model    observable   rmse_AU  rmse_over_range
0    Miller 2026 MLE  scaled_pSOS1  0.308809         0.030881
1    Miller 2026 MLE  scaled_pEGFR  1.528698         0.179847
2    Miller 2026 MLE   scaled_pERK  1.783538         0.198171
3  Kocieniewski 2013  scaled_pSOS1  1.444038         0.144404
4  Kocieniewski 2013  scaled_pEGFR  1.143504         0.134530
5  Kocieniewski 2013   scaled_pERK  1.681197         0.186800


In [5]:
kocieniewski_digitized = pd.read_csv(
    REF_DIR / "kocieniewski2013_figure4_wt_model_predictions_digitized.csv"
)

kocieniewski_bng = table_from_gdat(
    REF_DIR / "mek_isoform_erk_feedback_miller2026_kocieniewski2013_ode.gdat"
)
kocieniewski_bng["time_min"] = kocieniewski_bng["time"] / 60.0
kocieniewski_bng["MEK_pRDS_percent"] = kocieniewski_bng["MEK_pRDS"] / 200000.0 * 100.0
kocieniewski_bng["pERK1_2_percent"] = kocieniewski_bng["pERK1_2_wt"] / 3000000.0 * 100.0

kocieniewski_specs = {
    "MEK_pRDS_WT": {"observable": "MEK_pRDS_percent", "ymax": 80.0},
    "pERK1_2_WT": {"observable": "pERK1_2_percent", "ymax": 100.0},
}

kocieniewski_rows = []
for curve, spec in kocieniewski_specs.items():
    cloud = kocieniewski_digitized[
        (kocieniewski_digitized["curve"] == curve)
        & (kocieniewski_digitized["sample_type"].isin(["origin_anchor", "curve_pixel"]))
    ]
    center = kocieniewski_digitized[
        (kocieniewski_digitized["curve"] == curve)
        & (kocieniewski_digitized["sample_type"] == "centerline_by_x_median")
    ]
    tree = cKDTree(
        np.column_stack(
            [
                cloud["time_min"].to_numpy(dtype=float) / 60.0,
                cloud["value_percent"].to_numpy(dtype=float) / spec["ymax"],
            ]
        )
    )
    model = kocieniewski_bng[
        (kocieniewski_bng["time_min"] >= 0) & (kocieniewski_bng["time_min"] <= 60)
    ]
    model_points = np.column_stack(
        [
            model["time_min"].to_numpy(dtype=float) / 60.0,
            model[spec["observable"]].to_numpy(dtype=float) / spec["ymax"],
        ]
    )
    distances, _ = tree.query(model_points)

    center_peak = center.loc[center["value_percent"].idxmax()]
    model_peak = kocieniewski_bng.loc[kocieniewski_bng[spec["observable"]].idxmax()]
    center_60 = center.iloc[(center["time_min"] - 60.0).abs().argmin()]
    model_60 = np.interp(60.0, kocieniewski_bng["time_min"], kocieniewski_bng[spec["observable"]])

    kocieniewski_rows.append(
        {
            "model": "Kocieniewski 2013",
            "reported_source": "Kocieniewski and Lipniacki 2013 Figure 4 lower WT curves",
            "curve": curve,
            "model_points": int(len(model)),
            "digitized_cloud_points": int(len(cloud)),
            "median_nearest_curve_distance": float(np.median(distances)),
            "p95_nearest_curve_distance": float(np.percentile(distances, 95)),
            "max_nearest_curve_distance": float(np.max(distances)),
            "peak_time_abs_error_min": float(abs(model_peak["time_min"] - center_peak["time_min"])),
            "peak_percent_abs_error": float(abs(model_peak[spec["observable"]] - center_peak["value_percent"])),
            "value_60_abs_error_percent": float(abs(model_60 - center_60["value_percent"])),
        }
    )

kocieniewski_reported_check = pd.DataFrame(kocieniewski_rows)
kocieniewski_reported_check


Out[0]: 
               model  ... value_60_abs_error_percent
0  Kocieniewski 2013  ...                   0.172587
1  Kocieniewski 2013  ...                   0.411418

[2 rows x 11 columns]


In [6]:
assert reference_check["ode_net_exact"].all()
assert (reference_check["gdat_max_rel_or_abs"] <= 1e-12).all()
assert (reference_check["cdat_max_rel_or_abs"] <= 1e-12).all()
assert (independent_check["max_rel_or_abs"] <= 1e-5).all()

# Miller et al. data are arbitrary-unit densitometry measurements, so the
# tolerance is stated as RMSE over each reported trajectory range.
assert (reported_check["rmse_over_range"] <= 0.25).all()

# Kocieniewski and Lipniacki Figure 4 curves are digitized from a 300 dpi
# PDF rendering. Nearest-curve distances are normalized to the plotted
# x- and y-axis ranges and are robust to line thickness and the near-vertical
# EGF-response onset.
assert (kocieniewski_reported_check["p95_nearest_curve_distance"] <= 0.01).all()
assert (kocieniewski_reported_check["max_nearest_curve_distance"] <= 0.05).all()
assert (kocieniewski_reported_check["peak_time_abs_error_min"] <= 0.3).all()
assert (kocieniewski_reported_check["peak_percent_abs_error"] <= 1.0).all()

print("Reference regeneration, independent integration, and reported-data checks passed.")
print("Maximum independent-integration relative/absolute error:")
print(independent_check.groupby("model")["max_rel_or_abs"].max())


Reference regeneration, independent integration, and reported-data checks passed.
Maximum independent-integration relative/absolute error:
model
Kocieniewski 2013    6.950803e-07
Miller 2026 MLE      1.324675e-06
Name: max_rel_or_abs, dtype: float64


In [7]:
plot_specs = [
    ("scaled_pSOS1", "pSOS1 (AU)"),
    ("scaled_pEGFR", "pEGFR (AU)"),
    ("scaled_pERK", "pERK1/2 (AU)"),
]
colors = {
    "mek_isoform_erk_feedback_miller2026": "#1f77b4",
    "mek_isoform_erk_feedback_miller2026_kocieniewski2013": "#d62728",
}

fig, axes = plt.subplots(2, 3, figsize=(12, 7.2), constrained_layout=True)
for ax, (col, ylabel) in zip(axes[0], plot_specs):
    for stem in MODEL_STEMS:
        bng = table_from_gdat(REF_DIR / f"{stem}_ode.gdat")
        ax.plot(bng["time"] / 60.0, bng[col], label=LABELS[stem], color=colors[stem])
    ax.scatter(
        reported["time"] / 60.0,
        reported[col],
        s=28,
        facecolor="white",
        edgecolor="black",
        linewidth=1.0,
        zorder=3,
        label="WT data" if col == "scaled_pSOS1" else None,
    )
    ax.set_title(ylabel)
    ax.set_xlabel("time (min)")
    ax.grid(True, alpha=0.25)
axes[0, 0].set_ylabel("arbitrary units")
axes[0, 0].legend(frameon=False, fontsize=8)

figure4_plots = [
    ("MEK_pRDS_WT", "MEK pRDS (%)", "MEK_pRDS_percent"),
    ("pERK1_2_WT", "pERK1/2 (%)", "pERK1_2_percent"),
]
for ax, (curve, ylabel, col) in zip(axes[1, :2], figure4_plots):
    cloud = kocieniewski_digitized[
        (kocieniewski_digitized["curve"] == curve)
        & (kocieniewski_digitized["sample_type"] == "curve_pixel")
    ]
    center = kocieniewski_digitized[
        (kocieniewski_digitized["curve"] == curve)
        & (kocieniewski_digitized["sample_type"] == "centerline_by_x_median")
    ]
    ax.scatter(
        cloud["time_min"],
        cloud["value_percent"],
        s=1,
        color="#8c8c8c",
        alpha=0.18,
        linewidths=0,
        label="digitized curve pixels",
    )
    ax.plot(center["time_min"], center["value_percent"], color="black", linewidth=1.0, alpha=0.65)
    ax.plot(
        kocieniewski_bng["time_min"],
        kocieniewski_bng[col],
        color=colors["mek_isoform_erk_feedback_miller2026_kocieniewski2013"],
        linewidth=2.0,
        label="curated 2013 variant",
    )
    ax.set_title(ylabel)
    ax.set_xlabel("time (min)")
    ax.grid(True, alpha=0.25)
axes[1, 0].set_ylabel("percent of total")
axes[1, 0].legend(frameon=False, fontsize=8)
axes[1, 2].axis("off")

fig.suptitle("Reported-data comparisons", y=1.02)
fig.savefig(MODEL_DIR / "verify_miller2026.png", dpi=200, bbox_inches="tight")
plt.show()


<ipython-input-1-b8faa16eec60>:71: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

The notebook checks exact regeneration of BioNetGen reference outputs, independent SciPy integration of the generated mass-action network, quantitative agreement with the Miller et al. WT data table, and quantitative agreement with digitized Kocieniewski and Lipniacki (2013) Figure 4 WT model-prediction curves. The Miller comparison uses RMSE over each arbitrary-unit trajectory range; the 2013 comparison uses normalized nearest-curve distances plus peak timing/amplitude checks because the reported curves are thick plotted lines digitized from a 300 dpi PDF rendering.
